# GLIDE Output Visualization Starter

Use this notebook to inspect LPDM outputs (footprint Zarr and endpoint Parquet) and generate quick diagnostic plots.

## 1) Setup

If imports fail, install dependencies in your active environment:
- uv pip install -r requirements.txt

Then select the notebook kernel from your project environment.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import xarray as xr

# Ensure local package imports work when notebook is run from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from lpdm.visualize import (
    plot_3d_trajectories,
    plot_altitude_histogram_relative_to_blh,
)

print("Project root:", PROJECT_ROOT)

Project root: /Users/chxmr/code/glide


## 2) Configure Output Paths

Update these paths to point at the files written by your run.

In [2]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FOOTPRINT_ZARR = OUTPUT_DIR / "footprint.zarr"
ENDPOINTS_PARQUET = OUTPUT_DIR / "endpoints.parquet"

print("Footprint Zarr exists:", FOOTPRINT_ZARR.exists())
print("Endpoints Parquet exists:", ENDPOINTS_PARQUET.exists())

Footprint Zarr exists: False
Endpoints Parquet exists: False


## 3) Load Footprint and Endpoints

In [ ]:
fp_ds = xr.open_zarr(FOOTPRINT_ZARR) if FOOTPRINT_ZARR.exists() else None
endpoints_df = pd.read_parquet(ENDPOINTS_PARQUET) if ENDPOINTS_PARQUET.exists() else None

if fp_ds is not None:
    print(fp_ds)
else:
    print("Footprint Zarr not found.")

if endpoints_df is not None:
    display(endpoints_df.head())
    print("Rows:", len(endpoints_df))
else:
    print("Endpoints Parquet not found.")

## 4) Altitude Relative to BLH Histogram

Assumes endpoint columns include `z` for altitude in meters and optionally `blh` in meters.
If `blh` is not present, a scalar fallback can be used.

In [ ]:
if endpoints_df is not None and "z" in endpoints_df.columns:
    altitude_m = endpoints_df["z"].to_numpy()
    if "blh" in endpoints_df.columns:
        blh_m = endpoints_df["blh"].to_numpy()
    else:
        blh_m = 1000.0

    fig, ax = plot_altitude_histogram_relative_to_blh(
        altitude_m=altitude_m,
        blh_m=blh_m,
        bins=60,
        title="Endpoint Altitude Relative to BLH",
    )
else:
    print("Need endpoint data with a 'z' column to draw histogram.")

## 5) 3D Trajectory Plot (Plotly)

If you have full trajectory arrays, pass shape (T, N, 3).
This cell includes a synthetic example so plotting works immediately.

In [ ]:
# Synthetic fallback example trajectory cube: [time, particle, xyz]
T, N = 50, 40
t = np.linspace(0.0, 1.0, T)[:, None]
p = np.linspace(0.0, 2.0 * np.pi, N)[None, :]

x = 1000.0 * np.cos(2.0 * np.pi * t + p)
y = 1000.0 * np.sin(2.0 * np.pi * t + p)
z = 500.0 + 200.0 * np.sin(4.0 * np.pi * t + p)
traj = np.stack([x, y, z], axis=2)

fig = plot_3d_trajectories(traj, max_particles=40, title="Synthetic LPDM Trajectories")
fig.show()

## 6) Optional: Footprint Slice Quicklook

Attempts to display a 2D slice from the first time bin and first vertical bin.

In [ ]:
if fp_ds is not None:
    # Change variable name if your dataset uses a different one.
    candidate_vars = list(fp_ds.data_vars)
    if not candidate_vars:
        print("No data variables found in footprint dataset.")
    else:
        var_name = candidate_vars[0]
        da = fp_ds[var_name]
        print("Using variable:", var_name, "shape:", da.shape)

        # Expected dims like [time_ago, z_bin, y, x].
        slice_2d = da.isel({da.dims[0]: 0, da.dims[1]: 0})
        slice_2d.plot(cmap="viridis")
else:
    print("Footprint dataset not loaded.")